In [0]:
!pip install feature_engine

In [0]:
import mlflow

# IMPORTANDO MODELO
#model = mlflow.sklearn.load_model("models:/workspace.analytics.tmw_ativacao_t5/1")
model = mlflow.sklearn.load_model(
    "models:/m-df2d7992d08e42c2a690ae2bca8ad3be"
)

# IMPORTANDO DADOS
df = spark.sql(""" 
    SELECT *
            EXCEPT (t2.dtRef, t2.idCliente)
    FROM workspace.analytics.fs_pontos AS t1

    LEFT JOIN workspace.analytics.fs_cursos AS t2
    ON t1.IdCliente = t2.idCliente
    AND t1.dtRef = t2.dtRef

    WHERE t1.dtRef = (select max(dtRef) from workspace.analytics.fs_pontos)

""").toPandas()



In [0]:
X = df[model.feature_names_in_]

print(X.shape)

In [0]:
X = df[model.feature_names_in_]
# Probabilidade de churn
predict = model.predict_proba(X)[:, 0]

# Adiciona a probabilidade à base
df['proba_churn'] = predict

# Ordena da maior para a menor probabilidade
df_churn = df[['IdCliente', 'proba_churn']].sort_values(
    by='proba_churn',
    ascending=False
)

# Seleciona os 50 clientes com maior probabilidade de churn
top_50_churn = df_churn.head(50)

display(top_50_churn)